# 06 — Full MAPPO Training with Curriculum Progression

20,000-episode training run with curriculum learning through geometry tiers:
- **Phase 1 (Easy)**: Tier 0 open fields, 5–15 agents
- **Phase 2 (Medium)**: Tier 0–1 corridors/bottlenecks, 10–30 agents
- **Phase 3 (Hard)**: Tier 1–2 branching corridors, 20–50 agents
- **Phase 4 (Full)**: All tiers, 20–100 agents

Rewards include inverse-distance-to-goal shaping for better intermediate signal.
Episodes use 1000 steps (100s sim time) to give agents time to navigate complex geometries.

In [ ]:
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import cm
from collections import defaultdict

from crowdrl_env.crowd_env import CrowdEnv, CrowdEnvConfig
from crowdrl_env.geometry_generator import GeometryTier, GeometryConfig
from crowdrl_env.reward import RewardConfig
from crowdrl_env.spawner import SpawnConfig
from crowdrl_env.visualiser import plot_geometry

from crowdrl_train.buffer import RolloutBuffer
from crowdrl_train.config import (
    NetworkConfig, PPOConfig, CurriculumConfig, CurriculumPhase, TrainConfig,
)
from crowdrl_train.curriculum import CurriculumManager, EpisodeStats
from crowdrl_train.mappo import MAPPOUpdater
from crowdrl_train.networks import ActorCritic
from crowdrl_train.normalizer import RunningNormalizer, RewardNormalizer
from crowdrl_train.train import collect_episode

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__}, device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name()}")

## 1. Training Configuration

Key choices:
- **Inverse distance reward** (`inverse_distance_weight=0.05`): continuous proximity signal
- **Progress weight** raised to 0.5 for stronger directional shaping
- **max_steps=1000**: 100s sim time — enough for agents at 1.3 m/s to traverse 130m paths
- **Curriculum**: 4 phases, advance when rolling goal rate exceeds threshold
- **min_episodes_per_phase=500**: ensure sufficient training per difficulty level

In [ ]:
# Curriculum phases — progressive difficulty
phases = (
    CurriculumPhase(
        name="easy",
        geometry_tiers=(GeometryTier.TIER_0,),
        n_agents_range=(5, 15),
        goal_rate_threshold=0.70,
    ),
    CurriculumPhase(
        name="medium",
        geometry_tiers=(GeometryTier.TIER_0, GeometryTier.TIER_1),
        n_agents_range=(10, 30),
        goal_rate_threshold=0.60,
    ),
    CurriculumPhase(
        name="hard",
        geometry_tiers=(GeometryTier.TIER_1, GeometryTier.TIER_2),
        n_agents_range=(20, 50),
        goal_rate_threshold=0.50,
    ),
    CurriculumPhase(
        name="full",
        geometry_tiers=(GeometryTier.TIER_0, GeometryTier.TIER_1, GeometryTier.TIER_2),
        n_agents_range=(20, 100),
        goal_rate_threshold=0.0,  # terminal phase
    ),
)

curriculum_config = CurriculumConfig(
    phases=phases,
    metric_window=100,
    min_episodes_per_phase=500,
)

# Reward with inverse distance shaping
reward_config = RewardConfig(
    goal_bonus=10.0,
    collision_penalty=-1.0,
    timeout_penalty=-5.0,
    progress_weight=0.5,
    inverse_distance_weight=0.05,
    use_smoothness=True,
)

# Environment base config — curriculum overrides tiers and agent count
env_config = CrowdEnvConfig(
    reward=reward_config,
    max_steps=1000,  # 100s sim time at dt=0.1
)

# Network — (256, 256) for full training
net_config = NetworkConfig(
    obs_dim=env_config.obs.obs_dim,
    action_dim=env_config.action.action_dim,
    actor_hidden_sizes=(256, 256),
    critic_hidden_sizes=(256, 256),
)

ppo_config = PPOConfig(
    lr_actor=5e-4,
    lr_critic=5e-4,
    n_epochs=10,
    clip_epsilon=0.2,
    gamma=0.99,
    gae_lambda=0.95,
    target_kl=0.02,
    lr_schedule="linear",
)

SEED = 42
N_EPISODES = 20_000
LOG_INTERVAL = 50
CHECKPOINT_INTERVAL = 1000

print(f"Target: {N_EPISODES} episodes")
print(f"Curriculum: {' -> '.join(p.name for p in phases)}")
print(f"max_steps={env_config.max_steps} ({env_config.max_steps * env_config.dt:.0f}s sim time)")
print(f"Reward: progress={reward_config.progress_weight}, "
      f"inv_dist={reward_config.inverse_distance_weight}, "
      f"goal={reward_config.goal_bonus}")

## 2. Initialise Training Components

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

actor_critic = ActorCritic(net_config).to(device)
updater = MAPPOUpdater(actor_critic, ppo_config, device)
buffer = RolloutBuffer(net_config.obs_dim, net_config.action_dim, device)
curriculum = CurriculumManager(curriculum_config)

obs_normalizer = RunningNormalizer(shape=(net_config.obs_dim,))
reward_normalizer = RewardNormalizer(gamma=ppo_config.gamma)

# Create initial env from curriculum
cur_env_config = curriculum.make_env_config(env_config)
env = CrowdEnv(config=cur_env_config, seed=SEED)

n_params = sum(p.numel() for p in actor_critic.parameters())
print(f"Actor-Critic: {n_params:,} parameters")
print(f"Starting phase: {curriculum.current_phase.name}")

## 3. Training Loop

Collects one episode per iteration, runs PPO update, tracks curriculum advancement.

In [ ]:
# Training history
history = defaultdict(list)
phase_transitions = []  # (episode_idx, phase_name)
total_agent_steps = 0
start_time = time.time()

print(f"{'Ep':>6} | {'Steps':>12} | {'GoalRate':>8} | {'Reward':>8} | "
      f"{'Agents':>6} | {'EpLen':>5} | {'Phase':>8} | {'SPS':>8}")
print("-" * 85)

for ep in range(1, N_EPISODES + 1):
    # Collect episode
    ep_stats = collect_episode(
        env, actor_critic, buffer, obs_normalizer, reward_normalizer, device
    )

    # GAE
    n_agents = ep_stats["n_agents"]
    last_values = np.zeros(n_agents, dtype=np.float64)
    last_dones = np.ones(n_agents, dtype=np.bool_)
    buffer.compute_gae(last_values, last_dones, ppo_config.gamma, ppo_config.gae_lambda)

    # PPO update
    flat_batch = buffer.flatten()
    if flat_batch.batch_size > 0:
        update_metrics = updater.update(flat_batch)
    else:
        update_metrics = {}

    ep_agent_steps = buffer.total_active_agent_steps
    total_agent_steps += ep_agent_steps
    buffer.clear()

    # LR decay (linear over total episodes)
    progress = ep / N_EPISODES
    updater.update_learning_rate(progress)

    # Record history
    history["goal_rate"].append(ep_stats["goal_rate"])
    history["mean_reward"].append(ep_stats["mean_reward"])
    history["episode_length"].append(ep_stats["episode_length"])
    history["n_agents"].append(ep_stats["n_agents"])
    history["n_reached_goal"].append(ep_stats["n_reached_goal"])
    history["phase_idx"].append(curriculum.current_phase_idx)
    history["total_steps"].append(total_agent_steps)
    if update_metrics:
        history["policy_loss"].append(update_metrics.get("policy_loss", 0))
        history["value_loss"].append(update_metrics.get("value_loss", 0))
        history["entropy"].append(update_metrics.get("entropy", 0))
        history["approx_kl"].append(update_metrics.get("approx_kl", 0))

    # Curriculum update
    cur_stats = EpisodeStats(
        goal_rate=ep_stats["goal_rate"],
        n_agents=ep_stats["n_agents"],
        episode_length=ep_stats["episode_length"],
        mean_reward=ep_stats["mean_reward"],
    )
    phase_advanced = curriculum.report_episode(cur_stats)
    if phase_advanced:
        phase_transitions.append((ep, curriculum.current_phase.name))
        cur_env_config = curriculum.make_env_config(env_config)
        env = CrowdEnv(config=cur_env_config, seed=SEED + ep)
        print(f"\n>>> Phase advanced to: {curriculum.current_phase.name} (episode {ep})\n")

    # Periodic logging
    if ep % LOG_INTERVAL == 0:
        elapsed = time.time() - start_time
        sps = total_agent_steps / max(elapsed, 1)
        # Rolling averages
        window = min(100, ep)
        avg_gr = np.mean(history["goal_rate"][-window:])
        avg_rw = np.mean(history["mean_reward"][-window:])
        print(
            f"{ep:>6} | {total_agent_steps:>12,} | {avg_gr:>8.3f} | {avg_rw:>8.2f} | "
            f"{ep_stats['n_agents']:>6} | {ep_stats['episode_length']:>5} | "
            f"{curriculum.current_phase.name:>8} | {sps:>8.0f}"
        )

elapsed = time.time() - start_time
print(f"\nDone: {N_EPISODES} episodes, {total_agent_steps:,} agent-steps in {elapsed:.1f}s")
print(f"Phase transitions: {phase_transitions}")

## 4. Training Curves

In [ ]:
def smooth(values, window=100):
    """Rolling mean with edge handling."""
    if len(values) < window:
        return np.array(values)
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="valid")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Phase background shading helper
phase_colors = ["#e8f5e9", "#fff3e0", "#fce4ec", "#e3f2fd"]
phase_names_unique = ["easy", "medium", "hard", "full"]

def add_phase_bg(ax):
    """Add phase background shading to an axis."""
    prev_ep = 0
    for ep_idx, name in phase_transitions:
        pidx = phase_names_unique.index(name) - 1  # previous phase
        if 0 <= pidx < len(phase_colors):
            ax.axvspan(prev_ep, ep_idx, alpha=0.3, color=phase_colors[pidx])
        prev_ep = ep_idx
    # Final phase
    if phase_transitions:
        last_pidx = phase_names_unique.index(phase_transitions[-1][1])
        ax.axvspan(prev_ep, len(history["goal_rate"]), alpha=0.3, color=phase_colors[min(last_pidx, 3)])
    else:
        ax.axvspan(0, len(history["goal_rate"]), alpha=0.3, color=phase_colors[0])

# Goal rate
ax = axes[0, 0]
add_phase_bg(ax)
ax.plot(smooth(history["goal_rate"]), color="tab:green", linewidth=0.8)
ax.set_ylabel("Goal Rate")
ax.set_xlabel("Episode")
ax.set_title("Goal Rate (rolling 100)")
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)

# Mean reward
ax = axes[0, 1]
add_phase_bg(ax)
ax.plot(smooth(history["mean_reward"]), color="tab:blue", linewidth=0.8)
ax.set_ylabel("Mean Reward")
ax.set_xlabel("Episode")
ax.set_title("Mean Reward (rolling 100)")
ax.grid(True, alpha=0.3)

# Episode length
ax = axes[0, 2]
add_phase_bg(ax)
ax.plot(smooth(history["episode_length"]), color="tab:orange", linewidth=0.8)
ax.set_ylabel("Episode Length")
ax.set_xlabel("Episode")
ax.set_title("Episode Length (rolling 100)")
ax.grid(True, alpha=0.3)

# Policy + value loss
ax = axes[1, 0]
if history["policy_loss"]:
    ax.plot(smooth(history["policy_loss"]), color="tab:red", linewidth=0.8, label="Policy")
    ax2 = ax.twinx()
    ax2.plot(smooth(history["value_loss"]), color="tab:purple", linewidth=0.8, label="Value")
    ax2.set_ylabel("Value Loss", color="tab:purple")
    ax.legend(loc="upper left")
    ax2.legend(loc="upper right")
ax.set_ylabel("Policy Loss", color="tab:red")
ax.set_xlabel("Episode")
ax.set_title("Losses (rolling 100)")
ax.grid(True, alpha=0.3)

# Entropy
ax = axes[1, 1]
if history["entropy"]:
    ax.plot(smooth(history["entropy"]), color="tab:cyan", linewidth=0.8)
ax.set_ylabel("Entropy")
ax.set_xlabel("Episode")
ax.set_title("Policy Entropy (rolling 100)")
ax.grid(True, alpha=0.3)

# Agent count + curriculum phase
ax = axes[1, 2]
add_phase_bg(ax)
ax.plot(smooth(history["n_agents"]), color="tab:brown", linewidth=0.8)
ax.set_ylabel("Agent Count")
ax.set_xlabel("Episode")
ax.set_title("Agents per Episode (rolling 100)")
ax.grid(True, alpha=0.3)
# Add phase transition markers
for ep_idx, name in phase_transitions:
    ax.axvline(ep_idx, color="red", linestyle="--", alpha=0.7)
    ax.text(ep_idx, ax.get_ylim()[1] * 0.95, f" {name}", fontsize=8, color="red")

fig.suptitle("MAPPO Training — 20k Episodes with Curriculum", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Per-Phase Performance Summary

In [ ]:
# Aggregate stats per curriculum phase
phase_idx_arr = np.array(history["phase_idx"])
goal_rate_arr = np.array(history["goal_rate"])
reward_arr = np.array(history["mean_reward"])
length_arr = np.array(history["episode_length"])
agents_arr = np.array(history["n_agents"])

print(f"{'Phase':<10} {'Episodes':>8} {'GoalRate':>10} {'Reward':>10} {'EpLen':>8} {'Agents':>8}")
print("-" * 60)
for pidx, phase in enumerate(phases):
    mask = phase_idx_arr == pidx
    if not mask.any():
        continue
    n_eps = mask.sum()
    print(
        f"{phase.name:<10} {n_eps:>8} {goal_rate_arr[mask].mean():>10.3f} "
        f"{reward_arr[mask].mean():>10.2f} {length_arr[mask].mean():>8.1f} "
        f"{agents_arr[mask].mean():>8.1f}"
    )

# Final 500 episodes performance
print(f"\nFinal 500 episodes:")
print(f"  Goal rate: {np.mean(history['goal_rate'][-500:]):.3f}")
print(f"  Mean reward: {np.mean(history['mean_reward'][-500:]):.2f}")

## 6. Evaluation — Trajectory Visualisation

Run the trained policy on example geometries from each tier and plot agent trajectories.

In [ ]:
def run_episode_with_trajectories(env, actor_critic, obs_normalizer, device, max_steps=1000):
    """Run one episode collecting full agent trajectories."""
    obs, info = env.reset()
    n_agents = info["n_agents"]
    trajectories = [[] for _ in range(n_agents)]
    goal_positions = env._world.goal_positions.copy()
    polygon = env._world.walkable_polygon
    reached_goal = np.zeros(n_agents, dtype=bool)

    # Record starting positions
    for i in range(n_agents):
        trajectories[i].append(env._world.positions[i].copy())

    for step in range(max_steps):
        if obs_normalizer is not None:
            obs_norm = obs_normalizer.normalize(obs)
        else:
            obs_norm = obs

        with torch.no_grad():
            obs_t = torch.as_tensor(obs_norm, dtype=torch.float32, device=device)
            actions, _, _, _, _ = actor_critic.get_action_and_value(obs_t)

        obs, rewards, terminated, truncated, step_info = env.step(actions.cpu().numpy())

        for i in range(n_agents):
            trajectories[i].append(env._world.positions[i].copy())

        reached_goal |= terminated

        if step_info.get("episode_over", False):
            break

    return trajectories, goal_positions, polygon, reached_goal, info


# Run evaluation on each tier
eval_tiers = [
    ("Tier 0 — Open field", GeometryTier.TIER_0, (5, 15)),
    ("Tier 0 — Open field (dense)", GeometryTier.TIER_0, (20, 30)),
    ("Tier 1 — Corridor", GeometryTier.TIER_1, (10, 20)),
    ("Tier 1 — Bottleneck", GeometryTier.TIER_1, (15, 25)),
    ("Tier 2 — Branching", GeometryTier.TIER_2, (15, 25)),
    ("Tier 2 — Complex", GeometryTier.TIER_2, (25, 40)),
]

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
axes_flat = axes.flatten()

actor_critic.eval()

for idx, (title, tier, agent_range) in enumerate(eval_tiers):
    ax = axes_flat[idx]

    eval_env_config = CrowdEnvConfig(
        geometry=GeometryConfig(tier=tier),
        spawn=SpawnConfig(n_agents_range=agent_range),
        reward=reward_config,
        max_steps=1000,
    )
    eval_env = CrowdEnv(config=eval_env_config, seed=SEED + idx + 100)

    trajs, goals, polygon, reached, info = run_episode_with_trajectories(
        eval_env, actor_critic, obs_normalizer, device
    )

    # Plot geometry
    plot_geometry(polygon, ax=ax)

    # Plot trajectories
    n_agents = len(trajs)
    cmap = cm.get_cmap("tab20", n_agents)
    for i in range(n_agents):
        traj = np.array(trajs[i])
        color = cmap(i % 20)
        alpha = 0.8 if reached[i] else 0.3
        ax.plot(traj[:, 0], traj[:, 1], color=color, linewidth=0.8, alpha=alpha)
        ax.plot(traj[0, 0], traj[0, 1], "o", color=color, markersize=3, alpha=alpha)
        ax.plot(goals[i, 0], goals[i, 1], "*", color=color, markersize=6, alpha=alpha)

    goal_rate = reached.sum() / n_agents if n_agents > 0 else 0
    ax.set_title(f"{title}\n{n_agents} agents, {goal_rate:.0%} reached goal", fontsize=10)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.2)

actor_critic.train()

fig.suptitle("Trained Policy — Agent Trajectories", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Quantitative Evaluation

Run 50 evaluation episodes per tier to get statistically meaningful performance numbers.

In [ ]:
N_EVAL = 50
eval_results = {}

actor_critic.eval()

for tier in [GeometryTier.TIER_0, GeometryTier.TIER_1, GeometryTier.TIER_2]:
    tier_stats = {"goal_rate": [], "mean_reward": [], "episode_length": [], "n_agents": []}

    eval_env_config = CrowdEnvConfig(
        geometry=GeometryConfig(tier=tier),
        spawn=SpawnConfig(n_agents_range=(10, 40)),
        reward=reward_config,
        max_steps=1000,
    )
    eval_env = CrowdEnv(config=eval_env_config, seed=SEED + 1000)
    eval_buffer = RolloutBuffer(net_config.obs_dim, net_config.action_dim, device)

    for i in range(N_EVAL):
        ep_stats = collect_episode(
            eval_env, actor_critic, eval_buffer, obs_normalizer, None, device
        )
        eval_buffer.clear()
        tier_stats["goal_rate"].append(ep_stats["goal_rate"])
        tier_stats["mean_reward"].append(ep_stats["mean_reward"])
        tier_stats["episode_length"].append(ep_stats["episode_length"])
        tier_stats["n_agents"].append(ep_stats["n_agents"])

    eval_results[tier.name] = tier_stats

actor_critic.train()

# Print results
print(f"{'Tier':<10} {'GoalRate':>10} {'Reward':>10} {'EpLen':>8} {'Agents':>8}")
print("-" * 50)
for tier_name, stats in eval_results.items():
    print(
        f"{tier_name:<10} "
        f"{np.mean(stats['goal_rate']):>8.3f}±{np.std(stats['goal_rate']):.3f} "
        f"{np.mean(stats['mean_reward']):>8.2f}±{np.std(stats['mean_reward']):.2f} "
        f"{np.mean(stats['episode_length']):>6.0f}±{np.std(stats['episode_length']):.0f} "
        f"{np.mean(stats['n_agents']):>6.1f}"
    )

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
tier_names = list(eval_results.keys())
x = np.arange(len(tier_names))

for ax, metric, ylabel in zip(
    axes,
    ["goal_rate", "mean_reward", "episode_length"],
    ["Goal Rate", "Mean Reward", "Episode Length"],
):
    means = [np.mean(eval_results[t][metric]) for t in tier_names]
    stds = [np.std(eval_results[t][metric]) for t in tier_names]
    bars = ax.bar(x, means, yerr=stds, capsize=5, color=["#4CAF50", "#FF9800", "#F44336"])
    ax.set_xticks(x)
    ax.set_xticklabels(tier_names)
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.grid(True, alpha=0.3, axis="y")

fig.suptitle(f"Evaluation Performance ({N_EVAL} episodes per tier)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Export

Save the trained policy as ONNX for deployment in JuPedSim.

In [ ]:
from crowdrl_train.export import export_onnx
from pathlib import Path

onnx_path = Path("outputs/06_full_training/policy.onnx")
export_onnx(actor_critic.actor, obs_normalizer, onnx_path)
print(f"ONNX policy exported to: {onnx_path}")
print(f"File size: {onnx_path.stat().st_size / 1024:.1f} KB")

# Also save checkpoint
from crowdrl_train.train import save_checkpoint

ckpt_path = Path("outputs/06_full_training/checkpoint_final.pt")
save_checkpoint(
    ckpt_path, actor_critic, updater,
    obs_normalizer, reward_normalizer, curriculum,
    total_agent_steps, N_EPISODES,
)
print(f"Checkpoint saved to: {ckpt_path}")